# FEATURE ENGINERING & DATA CLEANING

In [5]:
import pandas as pd

In [6]:
df= pd.read_csv('../data/raw/GlobalLandTemperaturesByCountry.csv')


In [7]:
df['dt']=pd.to_datetime(df['dt'])

df['year']=df['dt'].dt.year
df['month']=df['dt'].dt.month
df['day']=df['dt'].dt.day

df.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,Country,year,month,day
0,1743-11-01,4.384,2.294,Åland,1743,11,1
1,1743-12-01,NaN,NaN,Åland,1743,12,1
2,1744-01-01,NaN,NaN,Åland,1744,1,1
3,1744-02-01,NaN,NaN,Åland,1744,2,1
4,1744-03-01,NaN,NaN,Åland,1744,3,1


In [8]:
df_clean = df[df['year'] >= 1900].copy()

# 2. A maradék minimális hiányzó értéket töröljük
df_clean = df_clean.dropna(subset=['AverageTemperature'])
df_clean = df_clean.dropna(subset=['AverageTemperatureUncertainty'])

df_clean.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,Country,year,month,day
1874,1900-01-01,-3.026,0.538,Åland,1900,1,1
1875,1900-02-01,-8.063,0.657,Åland,1900,2,1
1876,1900-03-01,-3.196,0.467,Åland,1900,3,1
1877,1900-04-01,0.781,0.224,Åland,1900,4,1
1878,1900-05-01,4.960,0.503,Åland,1900,5,1


In [9]:
df_clean.isnull().sum()

dt                               0
AverageTemperature               0
AverageTemperatureUncertainty    0
Country                          0
year                             0
month                            0
day                              0
dtype: int64

In [10]:
hun_temp = df_clean[df_clean['Country'] == 'Hungary'].copy()

season_map = {
    12: 'Winter',
    1: 'Winter',
    2: 'Winter',
    3: 'Spring',
    4: 'Spring',
    5: 'Spring',
    6: 'Summer',
    7: 'Summer',
    8: 'Summer',
    9: 'Autumn',
    10: 'Autumn',
    11: 'Autumn',
}
hun_temp['season'] = hun_temp['month'].map(season_map)

# 3. Feature Engineering: Átlag feletti/alatti bináris indikátor (1/0)
mean_temp = hun_temp['AverageTemperature'].mean()
hun_temp['is_above_mean'] = (
    hun_temp['AverageTemperature'] > mean_temp
).astype(int)

hun_temp.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,Country,year,month,day,season,is_above_mean
239091,1900-01-01,0.325,0.845,Hungary,1900,1,1,Winter,0
239092,1900-02-01,3.874,0.971,Hungary,1900,2,1,Winter,0
239093,1900-03-01,2.063,0.504,Hungary,1900,3,1,Spring,0
239094,1900-04-01,9.030,0.522,Hungary,1900,4,1,Spring,0
239095,1900-05-01,14.282,0.243,Hungary,1900,5,1,Spring,1


In [11]:
# 1. Biztosítjuk a helyes időrendi sorrendet
hun_temp = hun_temp.sort_values('dt').reset_index(drop=True)

# 2. Lag Features (Késleltetett jellemzők) létrehozása
# Előző hónap hőmérséklete (1 hónapos eltolás)
hun_temp['temp_lag_1'] = hun_temp['AverageTemperature'].shift(1)

# Előző év azonos hónapjának hőmérséklete (12 hónapos eltolás)
hun_temp['temp_lag_12'] = hun_temp['AverageTemperature'].shift(12)

# Gördülő átlag: Az elmúlt 3 hónap átlaghőmérséklete
hun_temp['temp_roll_mean_3'] = (
    hun_temp['AverageTemperature'].shift(1).rolling(window=3).mean()
)

# 3. A legelső sorokban keletkező NaN értékek eltávolítása
# (Mivel a legelső soroknak nincs előzményük, a shift miatt NaN kerül oda)
hun_temp_ml = hun_temp.dropna().copy()

# Ellenőrzés
print(
    hun_temp_ml[
        [
            'dt',
            'AverageTemperature',
            'temp_lag_1',
            'temp_lag_12',
            'temp_roll_mean_3',
        ]
    ].head()
)

           dt  AverageTemperature  temp_lag_1  temp_lag_12  temp_roll_mean_3
12 1901-01-01              -6.181       0.732        0.325          6.051333
13 1901-02-01              -3.524      -6.181        3.874          0.404667
14 1901-03-01               5.078      -3.524        2.063         -2.991000
15 1901-04-01              10.005       5.078        9.030         -1.542333
16 1901-05-01              15.464      10.005       14.282          3.853000


In [15]:
hun_temp['dt'] = pd.to_datetime(hun_temp['dt'])
hun_temp = hun_temp.sort_values('dt').reset_index(drop=True)

# 2. Helyes Lag és Rolling oszlopok kiszámítása
# Az előző hónap hőmérséklete
hun_temp['temp_lag_1'] = hun_temp['AverageTemperature'].shift(1)

# Pontosan 12 hónappal ezelőtti hőmérséklet (előző év ugyanaz a hónapja)
hun_temp['temp_lag_12'] = hun_temp['AverageTemperature'].shift(12)

# Az ELMÚLT 3 hónap átlaga (a shift(1) miatt a tárgyhót NEM számoljuk bele!)
hun_temp['temp_roll_mean_3'] = (
    hun_temp['AverageTemperature'].shift(1).rolling(window=3).mean()
)

# 3. Az első 12 sor (a keletkezett NaN-ok) eldobása
hun_temp_ml = hun_temp.dropna().reset_index(drop=True)

# Ellenőrzés: Az 1901-01-01 sorában a temp_lag_12-nek pontosan 0.325-nek kell lennie,
# a temp_roll_mean_3-nak pedig a téli hónapok negatív/alacsony átlagát kell mutatnia!
hun_temp_ml.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,Country,year,month,day,season,is_above_mean,temp_lag_1,temp_lag_12,temp_roll_mean_3
0,1902-01-01,1.537,0.765,Hungary,1902,1,1,Winter,0,2.711,-6.181,5.546667
1,1902-02-01,1.803,0.662,Hungary,1902,2,1,Winter,0,1.537,-3.524,2.353667
2,1902-03-01,4.239,0.314,Hungary,1902,3,1,Spring,0,1.803,5.078,2.017000
3,1902-04-01,9.152,0.288,Hungary,1902,4,1,Spring,0,4.239,10.005,2.526333
4,1902-05-01,11.555,0.483,Hungary,1902,5,1,Spring,1,9.152,15.464,5.064667


In [16]:
hun_temp = hun_temp.dropna().reset_index(drop=True)

In [17]:
hun_temp.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,Country,year,month,day,season,is_above_mean,temp_lag_1,temp_lag_12,temp_roll_mean_3
0,1902-01-01,1.537,0.765,Hungary,1902,1,1,Winter,0,2.711,-6.181,5.546667
1,1902-02-01,1.803,0.662,Hungary,1902,2,1,Winter,0,1.537,-3.524,2.353667
2,1902-03-01,4.239,0.314,Hungary,1902,3,1,Spring,0,1.803,5.078,2.017000
3,1902-04-01,9.152,0.288,Hungary,1902,4,1,Spring,0,4.239,10.005,2.526333
4,1902-05-01,11.555,0.483,Hungary,1902,5,1,Spring,1,9.152,15.464,5.064667


In [19]:
hun_temp.to_csv('../data/processed/hungary_temperatures_processed.csv', index=False)